In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("01_Ingestion")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")

    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .config(
        "spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version",
        "2"
    )

    .config(
        "spark.sql.sources.commitProtocolClass",
        "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol"
    )

    .config(
        "spark.sql.parquet.output.committer.class",
        "org.apache.parquet.hadoop.ParquetOutputCommitter"
    )

    .getOrCreate()
)
spark

In [2]:
df = (
    spark.read
    .option("header", True)
    .csv("/home/jovyan/data/online_retail_II.csv")
)

print(df.count())

df.show(5)

1067371
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
only showing top 5 rows



In [3]:
df.printSchema()

df.show(5)

print("Rows:", df.count())

print("Columns:", len(df.columns))

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Country: string (nullable = true)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      48|20

In [4]:
print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

Rows    : 1,067,371
Columns : 8


In [5]:
df.show(10, truncate=False)

+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|Customer ID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|489434 |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085.0    |United Kingdom|
|489434 |79323P   |PINK CHERRY LIGHTS                 |12      |2009-12-01 07:45:00|6.75 |13085.0    |United Kingdom|
|489434 |79323W   | WHITE CHERRY LIGHTS               |12      |2009-12-01 07:45:00|6.75 |13085.0    |United Kingdom|
|489434 |22041    |"RECORD FRAME 7"" SINGLE SIZE "    |48      |2009-12-01 07:45:00|2.1  |13085.0    |United Kingdom|
|489434 |21232    |STRAWBERRY CERAMIC TRINKET BOX     |24      |2009-12-01 07:45:00|1.25 |13085.0    |United Kingdom|
|489434 |22064    |PINK DOUGHNUT TRINKET POT          |2

In [6]:
df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Country: string (nullable = true)



In [7]:
df.describe().show()

+-------+------------------+------------------+--------------------+-----------------+-------------------+------------------+------------------+-----------+
|summary|           Invoice|         StockCode|         Description|         Quantity|        InvoiceDate|             Price|       Customer ID|    Country|
+-------+------------------+------------------+--------------------+-----------------+-------------------+------------------+------------------+-----------+
|  count|           1067371|           1067371|             1062989|          1067371|            1067371|           1067371|            824364|    1067371|
|   mean| 537608.1499316233|29011.161534536903|            21848.25|  9.9388984711033|               NULL|  4.64938772741731| 15324.63850435002|       NULL|
| stddev|26662.450446905917|18822.942866189227|   922.9197780233488|172.7057940767535|               NULL|123.55305872146349|1697.4644503793106|       NULL|
|    min|            489434|             10002|  DOORMAT U

In [8]:
missing = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)

    for c in df.columns
])

missing.show()

+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|Customer ID|Country|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|      0|        0|       4382|       0|          0|    0|     243007|      0|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+



In [9]:
total = df.count()

missing_percent = df.select([
    (
        count(
            when(col(c).isNull(), c)
        ) / total * 100
    ).alias(c)

    for c in df.columns
])

missing_percent.show()

+-------+---------+------------------+--------+-----------+-----+------------------+-------+
|Invoice|StockCode|       Description|Quantity|InvoiceDate|Price|       Customer ID|Country|
+-------+---------+------------------+--------+-----------+-----+------------------+-------+
|    0.0|      0.0|0.4105414143723223|     0.0|        0.0|  0.0|22.766872999172733|    0.0|
+-------+---------+------------------+--------+-----------+-----+------------------+-------+



In [10]:
df.select("Invoice").distinct().count()

53628

In [11]:
df.select("Customer ID").distinct().count()

5943

In [12]:
df.select("Country").distinct().show(50, False)

+--------------------+
|Country             |
+--------------------+
|Sweden              |
|Germany             |
|France              |
|Greece              |
|Belgium             |
|Finland             |
|Nigeria             |
|Unspecified         |
|Italy               |
|EIRE                |
|Norway              |
|Spain               |
|Denmark             |
|Channel Islands     |
|USA                 |
|Cyprus              |
|Switzerland         |
|United Arab Emirates|
|Japan               |
|Poland              |
|Portugal            |
|Australia           |
|Austria             |
|United Kingdom      |
|Netherlands         |
|RSA                 |
|Malta               |
|Bermuda             |
|Bahrain             |
|Singapore           |
|Hong Kong           |
|Lithuania           |
|Thailand            |
|Israel              |
|West Indies         |
|Lebanon             |
|Korea               |
|Canada              |
|Brazil              |
|Iceland             |
|Saudi Arab

In [13]:
df.select("Country").distinct().count()

43

In [14]:
print("Rows:", df.count())

print("Distinct Rows:", df.distinct().count())

Rows: 1067371
Distinct Rows: 1033036


In [15]:
spark.stop()

In [ ]:
(
    df.write
    .mode("overwrite")
    .option("header", True)
    .csv("s3a://bronze/online_retail_II.csv")
)
print("Data ingested to Bronze successfully!")